# Practice : Open-Source sLLM 실행 및 비교

[Task]
1. Hugging Face에서 sLLM을 직접 다운로드하여 실행
2. 동일 프롬프트에 대한 범용 LLM(ChatGPT, Claude 등) / Base 모델 / Instruct 모델의 응답 비교
3. 특정 업무(Text-to-SQL)에 특화 튜닝된 모델의 출력 확인

[Model]
- Base / Instruct 비교: Qwen2.5-1.5B, Qwen2.5-1.5B-Instruct
- 특화 튜닝 확인: SLM-SQL-0.5B

[Note]
- Colab 환경을 기본으로 작성, 로컬 환경 실행 시 안내를 별도 표기
- 모두 공개(gated 아님) 모델이므로 별도 로그인 불필요

## 1. Colab GPU 환경 설정 [Colab 전용]

- 런타임 > 런타임 유형 변경 > 하드웨어 가속기: T4 GPU 선택 후 저장
- 로컬 환경 사용자는 이 섹션 건너뛰기
- 무료 Colab은 세션 시간과 GPU 사용량에 제한이 있음. 연결이 끊기면 런타임을 다시 시작하고 위 셀부터 재실행

In [1]:
# 변경: nvidia-smi는 Colab/NVIDIA 전용이므로 Mac에서도 오류 없이 환경을 확인하도록 변경
import platform

print("OS:", platform.system())
print("CPU 아키텍처:", platform.machine())
print("NVIDIA GPU 확인은 Colab/Linux에서만 nvidia-smi를 사용하세요.")

OS: Darwin
CPU 아키텍처: arm64
NVIDIA GPU 확인은 Colab/Linux에서만 nvidia-smi를 사용하세요.


In [2]:
import torch

# 변경: NVIDIA CUDA뿐 아니라 Apple Silicon GPU(MPS)도 확인
print("CUDA 사용 가능:", torch.cuda.is_available())
print("Apple MPS 사용 가능:", torch.backends.mps.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    print("Device: Apple Silicon GPU (MPS)")
else:
    print("Device: CPU")

CUDA 사용 가능: False
Apple MPS 사용 가능: True
Device: Apple Silicon GPU (MPS)


## 2. 라이브러리 설치

- transformers 최신 버전 필요 (구버전 사용 시 모델 설정 인식 오류 발생 가능)
- 로컬 환경에서 이미 설치되어 있다면 생략 가능

In [3]:
# 변경: 현재 노트북 커널에 설치되도록 !pip 대신 %pip 사용
%pip install -U transformers accelerate


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 3. 라이브러리 Import

In [4]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(42)

# 변경: CUDA -> Apple MPS -> CPU 순서로 사용 가능한 연산 장치를 자동 선택
if torch.cuda.is_available():
    device = torch.device("cuda")
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    compute_dtype = torch.float16
else:
    device = torch.device("cpu")
    compute_dtype = torch.float32

print("Transformers version:", transformers.__version__)
print("사용 장치:", device)
print("Compute dtype:", compute_dtype)

/Users/beeyaaa/Workspace/SK-SKALA/0818 sLLM 구현 및 Fine Tunning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers version: 5.15.0
사용 장치: mps
Compute dtype: torch.float16


[Note] 모델 다운로드 저장 위치
- Hugging Face 캐시 경로"~/.cache/huggingface/hub"에 저장됨
- Colab은 런타임(가상 인스턴스)의 로컬 디스크에 저장 : 런타임 종료 시 캐시도 함께 삭제되어 재접속 시 재다운로드 필요
- 로컬 환경은 컴퓨터에 캐시가 계속 남아있어 재실행 시 다운로드 없이 바로 로드됨

## 4. Base 모델 로드 (Qwen2.5-1.5B)

- Instruction Tuning이 적용되지 않은 사전학습 모델
- Chat Template 없이 순수 텍스트 이어쓰기 방식으로 동작

In [5]:
base_model_id = "Qwen/Qwen2.5-1.5B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_id)
# 변경: device_map="auto" 대신 선택된 CUDA/MPS/CPU 장치로 명시적 이동
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    dtype=compute_dtype
).to(device)

print("Base 모델 로드 완료:", base_model_id)

Loading weights: 100%|██████████| 338/338 [00:03<00:00, 84.86it/s] 


Base 모델 로드 완료: Qwen/Qwen2.5-1.5B


## 5. Instruct 모델 로드 (Qwen2.5-1.5B-Instruct)

- Instruction Tuning이 적용된 모델
- Chat Template을 적용하여 대화 형식으로 입력

In [6]:
instruct_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_id)
# 변경: device_map="auto" 대신 선택된 CUDA/MPS/CPU 장치로 명시적 이동
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_id,
    dtype=compute_dtype
).to(device)

print("Instruct 모델 로드 완료:", instruct_model_id)

Loading weights: 100%|██████████| 338/338 [00:03<00:00, 92.63it/s] 


Instruct 모델 로드 완료: Qwen/Qwen2.5-1.5B-Instruct


## 6. 추론 함수 정의

- Base: 프롬프트를 그대로 이어쓰기
- Instruct: Chat Template 적용 후 생성

[Note] Chat Template ?
- Instruct 모델은 "<|im_start|>user ... <|im_end|>"처럼 발화자를 구분하는 특수 토큰 구조로 학습됨
- apply_chat_template()은 이 구조를 자동으로 만들어주는 함수이며, Base 모델은 이런 구조를 학습한 적이 없어 사용하지 않음

In [7]:
def generate_base(prompt: str, max_new_tokens: int = 200) -> str:
    inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
    output_ids = base_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return base_tokenizer.decode(generated, skip_special_tokens=True)


def generate_instruct(prompt: str, max_new_tokens: int = 200) -> str:
    messages = [{"role": "user", "content": prompt}]
    # return_dict=True: transformers 버전에 따라 apply_chat_template이
    # 텐서 대신 BatchEncoding을 반환하는 경우가 있어 명시적으로 지정
    inputs = instruct_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(instruct_model.device)
    output_ids = instruct_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return instruct_tokenizer.decode(generated, skip_special_tokens=True)

## 7. 비교 프롬프트 세트

[Note]
- 범용 LLM(ChatGPT, Claude 등) 결과는 아래 프롬프트를 웹 UI에 직접 질의하여 별도로 기록
- 이 노트북에서는 sLLM(Base/Instruct)만 코드로 직접 비교

In [8]:
single_turn_prompts = {
    "1_사실질문": "세종대왕이 만든 문자는 무엇인가요?",
    "2_지시문형": (
        "다음 문장을 한 문장으로 요약해줘: "
        "'인공지능 기술의 발전으로 다양한 산업 분야에서 자동화가 가속화되고 있으며, "
        "특히 제조업과 금융업에서 그 변화가 두드러지게 나타나고 있다.'"
    ),
    "3_추론문제": (
        "한 반에 학생이 30명 있습니다. 이 중 안경을 쓴 학생은 12명이고, "
        "안경을 쓴 학생 중 여학생은 7명입니다. "
        "안경을 쓰지 않은 학생 중 남학생이 10명이라면, "
        "안경을 쓰지 않은 여학생은 몇 명인가요?"
    ),
}

for name, prompt in single_turn_prompts.items():
    print(f"[{name}] {prompt}\n")

[1_사실질문] 세종대왕이 만든 문자는 무엇인가요?

[2_지시문형] 다음 문장을 한 문장으로 요약해줘: '인공지능 기술의 발전으로 다양한 산업 분야에서 자동화가 가속화되고 있으며, 특히 제조업과 금융업에서 그 변화가 두드러지게 나타나고 있다.'

[3_추론문제] 한 반에 학생이 30명 있습니다. 이 중 안경을 쓴 학생은 12명이고, 안경을 쓴 학생 중 여학생은 7명입니다. 안경을 쓰지 않은 학생 중 남학생이 10명이라면, 안경을 쓰지 않은 여학생은 몇 명인가요?



## 8. 단일 턴 프롬프트 실행 및 비교

[Note]
- Base 모델은 종료 시점을 스스로 판단하지 못해 매번 max_new_tokens만큼 끝까지 생성 : Instruct보다 시간이 더 걸림


In [9]:
# 변경: Mac에서 MPS 사용 여부도 확인할 수 있도록 device 출력 유지
print("Base 모델 device:", base_model.device, "| dtype:", base_model.dtype)
print("Instruct 모델 device:", instruct_model.device, "| dtype:", instruct_model.dtype)

if base_model.device.type == "cpu":
    print("\nGPU 가속을 사용하지 못하고 CPU에서 실행 중")
elif base_model.device.type == "mps":
    print("\nApple Silicon GPU(MPS)에서 실행 중")


Base 모델 device: mps:0 | dtype: torch.float16
Instruct 모델 device: mps:0 | dtype: torch.float16

Apple Silicon GPU(MPS)에서 실행 중


In [12]:
for name, prompt in single_turn_prompts.items():
    print(f"=== {name} ===")
    print("[Prompt]", prompt)
    print("\n[Base]")
    print(generate_base(prompt))
    print("\n[Instruct]")
    print(generate_instruct(prompt))
    print("\n" + "=" * 60 + "\n")

=== 1_사실질문 ===
[Prompt] 세종대왕이 만든 문자는 무엇인가요?

[Base]
 세종대왕이 만든 문자는 '조선자유자치'입니다. 세종대왕은 조선의 문화를 발전시키고, 민주주의를 실현하기 위해 노력했습니다. 그는 조선의 문화를 발전시키기 위해 '조선자유자치'를 만들었습니다. 이는 조선의 문화를 발전시키고, 민주주의를 실현하기 위해 노력했습니다. 그는 조선의 문화를 발전시키기 위해 '조선자유자치'를 만들었습니다. 이는 조선의 문화를 발전시키고, 민주주의를 실현하기 위해 노력했습니다. 그는 조선의 문화를 발전시키기 위해 '조선자유자치'를 만들었습니다. 이는 조선의 문화를 발전시키고, 민주주의를 실현하기 위해 노력했습니다. 그는 조

[Instruct]
세종대왕은 조선의 세 번째 왕으로서, 1418년부터 1450년까지 재위했습니다. 그는 많은 업적을 남겼지만, 가장 유명한 것은 한글을 개발한 것입니다.

세종대왕은 1443년에 "중국어"를 기반으로 한 새로운 글자体系을 만들었습니다. 이 새로운 글자는 '한글'이라고 불리며, 이후 조선의 정식 언어로 사용되었습니다.

한글은 중국의 한자와 일본의 한글(일본의 한글)과 달리, 한글은 한자보다 더 간단하고 직관적이며, 각 글자가 단순히 모양만으로 인식할 수 있습니다. 이것은 한국 문화의 중요한 특징 중 하나입니다.

세종대왕의 노력 덕분에, 오늘날 세계에서 가장 널리


=== 2_지시문형 ===
[Prompt] 다음 문장을 한 문장으로 요약해줘: '인공지능 기술의 발전으로 다양한 산업 분야에서 자동화가 가속화되고 있으며, 특히 제조업과 금융업에서 그 변화가 두드러지게 나타나고 있다.'

[Base]
 이에 따라, '인공지능 기술의 발전은 기업의 경쟁력과 혁신을 강화하고, 산업의 혁신과 발전을 이끌어 낼 수 있다.'라고 말했다.
'인공지능 기술의 발전은 기업의 경쟁력과 혁신을 강화하고, 산업의 혁신과 발전을 이끌어 낼 수 있다.'라고 말했다.

다음 문장을 한 문장으로 요약해줘: '인공지능 기술의 발전으로 다양

## 9. 멀티턴 대화 비교

[Note]
- Base 모델은 대화 형식(Chat Template)을 학습하지 않았으므로, 직전 턴과 답변을 프롬프트에 이어붙이는 방식으로 실행
- Instruct 모델은 messages 리스트에 대화 히스토리를 누적하여 실행

In [14]:
turn1 = "제주도 여행 코스를 하나 추천해줘"
turn2 = "방금 그거, 1박 2일로 압축해줄 수 있어?"

# Base: 이전 턴과 응답을 프롬프트에 이어붙임
base_turn1_response = generate_base(turn1)
base_multiturn_prompt = f"{turn1}\n{base_turn1_response}\n{turn2}"
base_turn2_response = generate_base(base_multiturn_prompt)

print("[Base - Turn1]", base_turn1_response)
print("\n[Base - Turn2]", base_turn2_response)

[Base - Turn1] 요. 저는 여행을 좋아하지만, 제주도는 아직도 잘 모르고 있어요. 제주도 여행 코스를 하나 추천해줘요. 저는 여행을 좋아하지만, 제주도는 아직도 잘 모르고 있어요. 제주도 여행 코스를 하나 추천해줘요. 저는 여행을 좋아하지만, 제주도는 아직도 잘 모르고 있어요. 제주도 여행 코스를 하나 추천해줘요. 저는 여행을 좋아하지만, 제주도는 아직도 잘 모르고 있어요. 제주도 여행 코스를 하나 추천해줘요. 저는 여행을 좋아하지만, 제주도는 아직도 잘 모르고 있어요. 제주도 여행 코스를 하나 추천해줘요. 저는 여행을 좋아하지만, 제주도는 아직도 잘 모르고 있어요. 제주도 여행 코스를 하나

[Base - Turn2]  제주도 여행 코스를 하나 추천해줘요. 저는 여행을 좋아하지만, 제주도는 아직도 잘 모르고 있어요. 제주도 여행 코스를 하나 추천해줘요. 저는 여행을 좋아하지만, 제주도는 아직도 잘 모르고 있어요. 제주도 여행 코스를 하나 추천해줘요. 저는 여행을 좋아하지만, 제주도는 아직도 잘 모르고 있어요. 제주도 여행 코스를 하나 추천해줘요. 저는 여행을 좋아하지만, 제주도는 아직도 잘 모르고 있어요. 제주도 여행 코스를 하나 추천해줘요. 저는 여행을 좋아하지만, 제주도는 아직도 잘 모르고 있어요. 제주도 여행 코스를 하나 추천해줘요. 저는 여행을 좋아하지만, 제주도는 아직도 잘 모르


In [15]:
# Instruct: messages 리스트에 대화 히스토리 누적
messages = [{"role": "user", "content": turn1}]
inputs = instruct_tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to(instruct_model.device)
output_ids = instruct_model.generate(**inputs, max_new_tokens=400, do_sample=False)
instruct_turn1_response = instruct_tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
)

messages.append({"role": "assistant", "content": instruct_turn1_response})
messages.append({"role": "user", "content": turn2})

inputs = instruct_tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to(instruct_model.device)
output_ids = instruct_model.generate(**inputs, max_new_tokens=400, do_sample=False)
instruct_turn2_response = instruct_tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
)

print("[Instruct - Turn1]", instruct_turn1_response)
print("\n[Instruct - Turn2]", instruct_turn2_response)

[Instruct - Turn1] 제주도는 한국에서 가장 인기 있는 관광지 중 하나로, 다양한 명소와 풍경을 제공합니다. 아래는 제주도의 주요 관광지들을 기반으로 한 여행 코스입니다:

1. 제주시 - 제주국립공원 및 해변
2. 서귀포시 - 서귀포 섬과 해양 스포츠
3. 제주시 - 제주도 최고의 해수욕장인 '제주 해안선'
4. 제주시 - 제주도의 중심부에 위치한 '제주도성'
5. 서귀포시 - 서귀포 섬의 자연 경관과 해양 스포츠

이러한 곳들을 방문하여 제주도의 아름다움을 체험하세요. 또한, 제주도의 특별한 요리를 맛볼 수 있는 '제주도식 음식'도 추천드립니다.

[Instruct - Turn2] 물론이죠! 1박 2일로 압축된 제주도 여행 코스를 소개해 드리겠습니다.

1. 제주의 해변을 즐기는 날: 
   - 제주시 해변을 둘러보기
   - 제주국립공원을 방문하기

2. 서귀포 섬을 즐기는 날:
   - 서귀포 섬을 타고 다니기
   - 서귀포 해양 스포츠를 즐기기

이렇게 하면 1박 2일 동안 제주도의 아름다운 자연과 해변을 즐길 수 있습니다. 또한, 제주도의 특별한 음식을 맛볼 수도 있습니다.


## 10. 범용 LLM과의 비교 기록

[Task] 
- 위 프롬프트(1~4)를 ChatGPT, Claude 등 범용 LLM 웹 UI에 동일하게 질의
- 아래 표에 결과를 요약하여 정리

| 프롬프트 | 범용 LLM | sLLM-Base | sLLM-Instruct |
|---|---|---|---|
| 1. 사실 질문 | | | |
| 2. 지시문형 | | | |
| 3. 추론 문제 | | | |
| 4. 멀티턴 | | | |

# 특화 튜닝 모델 확인 (Text-to-SQL)

## 11. 개요

[Task]
- 특정 업무(Text-to-SQL)에 특화 튜닝된 모델의 응답을 직접 실행하여 확인
- baseline과의 실행 비교 없이, 특화 모델 하나의 출력만 확인

[Model]
- 특화 튜닝: SLM-SQL-0.5B (Qwen2.5-Coder-0.5B-Instruct 기반, SFT + RL 튜닝)

[Note: Model Description]

 - Qwen2.5-Coder-0.5B-Instruct(범용 코드 모델)를 Text-to-SQL 작업에 SFT + RL로 추가 튜닝한 모델
- BIRD 벤치마크 기준, 같은 0.5B 파라미터에서도 튜닝 전후 정확도 차이가 크게 발생

| 모델 상태 | Execution Accuracy |
|---|---|
| 튜닝 전 (일반 SFT만, Qwen2.5-Coder-0.5B-Instruct) | 42.13% |
| 튜닝 후 (SFT + RL, SLM-SQL-0.5B) | 65.31% |

- (출처: SLM-SQL 논문, arXiv:2507.22478)


## 12. SQL 모델 로드

In [16]:
sql_tuned_id = "cycloneboy/SLM-SQL-0.5B"

sql_tuned_tokenizer = AutoTokenizer.from_pretrained(sql_tuned_id)
# 변경: SQL 모델도 선택된 CUDA/MPS/CPU 장치로 명시적 이동
sql_tuned_model = AutoModelForCausalLM.from_pretrained(
    sql_tuned_id, dtype=compute_dtype
).to(device)

print("특화 SQL 모델 로드 완료:", sql_tuned_id)

Loading weights: 100%|██████████| 291/291 [00:01<00:00, 162.54it/s]


특화 SQL 모델 로드 완료: cycloneboy/SLM-SQL-0.5B


## 13. SQL 생성 프롬프트 정의

- 스키마 정보를 함께 제공하고, 자연어 질문을 SQL로 변환

In [17]:
sql_schema = """
employees(id, name, dept_id, salary)
departments(dept_id, dept_name, location)
"""
sql_question = "서울에 위치한 부서별 평균 급여를 높은 순으로 보여줘."

sql_prompt = f"""다음 스키마를 참고하여 질문에 맞는 SQL 쿼리만 작성하세요.

스키마: {sql_schema}
질문: {sql_question}
SQL:"""

print(sql_prompt)

다음 스키마를 참고하여 질문에 맞는 SQL 쿼리만 작성하세요.

스키마: 
employees(id, name, dept_id, salary)
departments(dept_id, dept_name, location)

질문: 서울에 위치한 부서별 평균 급여를 높은 순으로 보여줘.
SQL:


## 14. SQL 생성 실행

In [18]:
def generate_sql(tokenizer, model, prompt: str, max_new_tokens: int = 150) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False
    )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


tuned_sql = generate_sql(sql_tuned_tokenizer, sql_tuned_model, sql_prompt)

print("=== 특화 튜닝 모델: SLM-SQL-0.5B ===")
print(tuned_sql)

=== 특화 튜닝 모델: SLM-SQL-0.5B ===
SELECT 
    d.dept_name,
    AVG(e.salary) AS average_salary
FROM 
    employees e
INNER JOIN 
    departments d ON e.dept_id = d.dept_id
WHERE 
    d.location = '서울'
GROUP BY 
    d.dept_name
ORDER BY 
    average_salary DESC;


In [20]:
# SQL 특화 모델에 일반 사실 질문과 추론 문제를 입력해 범용 능력 확인

def generate_response(
    tokenizer,
    model,
    prompt: str,
    max_new_tokens: int = 150
) -> str:
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )


test_prompts = {
    "1_사실질문": "세종대왕이 만든 문자는 무엇인가요?",

    "2_추론문제": (
        "한 반에 학생이 30명 있습니다. "
        "이 중 안경을 쓴 학생은 12명이고, "
        "안경을 쓴 학생 중 여학생은 7명입니다. "
        "안경을 쓰지 않은 학생 중 남학생이 10명이라면, "
        "안경을 쓰지 않은 여학생은 몇 명인가요?"
    ),
}


for prompt_name, prompt in test_prompts.items():
    response = generate_response(
        sql_tuned_tokenizer,
        sql_tuned_model,
        prompt
    )

    print(f"=== SLM-SQL-0.5B - {prompt_name} ===")
    print("[Prompt]")
    print(prompt)

    print("\n[Response]")
    print(response)

    print("\n" + "=" * 60 + "\n")

=== SLM-SQL-0.5B - 1_사실질문 ===
[Prompt]
세종대왕이 만든 문자는 무엇인가요?

[Response]
세종대왕이 만든 문자는 'A'입니다.


=== SLM-SQL-0.5B - 2_추론문제 ===
[Prompt]
한 반에 학생이 30명 있습니다. 이 중 안경을 쓴 학생은 12명이고, 안경을 쓴 학생 중 여학생은 7명입니다. 안경을 쓰지 않은 학생 중 남학생이 10명이라면, 안경을 쓰지 않은 여학생은 몇 명인가요?

[Response]
### 문제 해결 과정

1. **안경을 쓴 학생의 수**:
   - 안경을 쓴 학생은 12명입니다.
   - 안경을 쓴 학생 중 여학생은 7명입니다.

2. **안경을 쓰지 않은 학생의 수**:
   - 안경을 쓴 학생은 10명입니다.

3. **남학생과 여학생의 합계**:
   - 안경을 쓴 학생과 여학생의 합계는 12 + 7 = 19명입니다.

4. **남학생과 여




# [로컬 전용 실습] 네트워크 연결 없이 실행하기

## 15. 오프라인 추론 확인

[Task]
- 위 셀들을 모두 실행하여 모델 다운로드가 완전히 끝난 것을 확인한 뒤 진행
- Wi-Fi 또는 랜선을 분리한 상태에서 아래 셀을 실행하여, 로컬에 저장된 sLLM이 인터넷 없이도 동작하는지 확인

[Note]
- Colab은 클라우드 가상 환경이라 이 실습을 수행할 수 없음 (로컬 환경 전용)
- Colab에서는 (네트워크가 연결되어 있다면) "캐시 재사용이 잘 되는지"만 확인됨
- local_files_only=True 옵션으로 캐시된 파일만 사용하도록 강제

In [19]:
# 네트워크를 분리한 뒤 이 셀을 실행
offline_tokenizer = AutoTokenizer.from_pretrained(
    instruct_model_id, local_files_only=True
)
offline_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_id,
    # 변경: 오프라인 모델도 로딩 후 선택된 CUDA/MPS/CPU 장치로 이동
    dtype=compute_dtype,
    local_files_only=True
).to(device)

messages = [{"role": "user", "content": "sLLM이 무엇인지 한 문장으로 설명해줘"}]
inputs = offline_tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to(offline_model.device)
output_ids = offline_model.generate(**inputs, max_new_tokens=100, do_sample=False)

print(offline_tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Loading weights: 100%|██████████| 338/338 [00:04<00:00, 68.69it/s]


SLLM은 "사람과 로봇의 합성 언어 모델"을 의미합니다.
